# Week 13, Day 2: Adaptive methods: AdaGrad, RMSProp and Adam

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w013/d2.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Implement Adam

Write `adam(grad, x0, lr, steps, b1=0.9, b2=0.999, eps=1e-8)` returning the array of iterates. Check its first step on a gradient whose entries differ by six orders of magnitude.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def adam(grad, x0, lr, steps, b1=0.9, b2=0.999, eps=1e-8):
    w = np.array(x0, dtype=float)
    m, v = np.zeros_like(w), np.zeros_like(w)
    history = [w.copy()]
    for k in range(1, steps + 1):
        g = grad(w)
        m = ...
        v = ...
        m_hat = ...
        v_hat = ...
        w = ...
        history.append(w.copy())
    return np.array(history)

print("first step:", adam(lambda w: np.array([1000.0, -0.001, 3.0]), np.zeros(3), 0.01, 1)[1])

In [ ]:
import numpy as np
def test_first_step():
    "Adam's first step moves every parameter by about lr, against the sign of its gradient, whatever the gradient's size"
    g = lambda w: np.array([1000.0, -0.001, 3.0])
    h = adam(g, np.zeros(3), 0.01, 1)
    assert np.allclose(h[1], [-0.01, 0.01, -0.01], atol=1e-6)

def test_converges():
    "Adam minimises a simple quadratic"
    h = adam(lambda w: 2 * (w - 3.0), np.zeros(2), 0.1, 1000)
    assert np.allclose(h[-1], 3.0, atol=1e-2)

test_first_step()
test_converges()
print("All checks passed")

### Exercise: Rescale the loss

On the badly scaled quadratic with curvatures 100 and 1, run 200 steps of Adam (learning rate 0.05) and of gradient descent (learning rate 0.009) from (1, 1), for the loss itself and for the loss multiplied by 1000. Store the four paths and compare.

In [ ]:
H = np.diag([100.0, 1.0])
g1 = lambda w: H @ w
g1000 = lambda w: 1000 * (H @ w)
def gd(grad, x0, lr, steps):
    w = np.array(x0, dtype=float); out = [w.copy()]
    for _ in range(steps):
        w = w - lr * grad(w)
        if np.max(np.abs(w)) > 1e6: break
        out.append(w.copy())
    out += [out[-1] * 0 + 1e6] * (steps + 1 - len(out))      # pad a diverged run
    return np.array(out)

path_adam_1, path_adam_1000 = ..., ...
path_gd_1, path_gd_1000 = ..., ...
print("largest difference between Adam paths:", np.max(np.abs(path_adam_1 - path_adam_1000)).round(6))
print("gradient descent with the rescaled loss ends at", path_gd_1000[-1])

In [ ]:
import numpy as np
def test_adam_invariant():
    "Adam's path is nearly unchanged when the loss is multiplied by 1000"
    assert np.max(np.abs(path_adam_1 - path_adam_1000)) < 1e-3

def test_gd_not():
    "gradient descent's path changes completely (here it diverges or jumps far)"
    assert np.max(np.abs(path_gd_1 - path_gd_1000)) > 1

test_adam_invariant()
test_gd_not()
print("All checks passed")

In [ ]:
rng = np.random.default_rng(2018)
N, T = 50, 1500
prefs = np.linspace(0, np.pi, N, endpoint=False)
theta = rng.uniform(0, np.pi, T)
labels = (np.cos(2 * (theta - np.pi / 4)) > 0).astype(float)
counts = rng.poisson((10 + 20 * np.exp(2 * (np.cos(2 * (theta[:, None] - prefs[None])) - 1))) * 0.2)
Z = np.column_stack([counts, np.ones(T)])          # raw counts, badly conditioned
def sigmoid(u): return 1 / (1 + np.exp(-u))
def xent(w): p = np.clip(sigmoid(Z @ w), 1e-12, 1 - 1e-12); return -np.mean(labels * np.log(p) + (1 - labels) * np.log(1 - p))
def grad(w): return Z.T @ (sigmoid(Z @ w) - labels) / T
lam_max = np.linalg.eigvalsh(Z.T @ Z / (4 * T)).max()

### Exercise: Gradient descent, momentum and Adam

Train the raw-count decoder for 300 steps with gradient descent (lr 1.9/λmax), momentum (same lr, β = 0.9, written as a short loop) and Adam with learning rates 0.003, 0.01, 0.03 and 0.1. Store the final losses in the dictionary `final` (keys 'GD' and 'momentum') and Adam's in `adam_by_lr`.

In [ ]:
lr = 1.9 / lam_max
w = np.zeros(N + 1)
for _ in range(300):
    w = w - lr * grad(w)
final = {"GD": xent(w)}

w, v = np.zeros(N + 1), np.zeros(N + 1)
for _ in range(300):
    ...
final["momentum"] = xent(w)

adam_by_lr = {a: ... for a in [0.003, 0.01, 0.03, 0.1]}
print({k: round(v_, 4) for k, v_ in final.items()}, " Adam:", {k: round(v_, 4) for k, v_ in adam_by_lr.items()})

In [ ]:
import numpy as np
def test_adam_best():
    "Adam reaches a lower loss than gradient descent after 300 steps for every learning rate tried"
    assert all(l < final["GD"] - 0.005 for l in adam_by_lr.values())

def test_robust():
    "Adam's final loss varies little across a 30-fold range of learning rates"
    vals = list(adam_by_lr.values())
    assert max(vals) - min(vals) < 0.01

test_adam_best()
test_robust()
print("All checks passed")